In [1]:
# 4. Design your function here! 

# For this demo, we do a basic NDVI calculation.
def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df

In [ ]:
# 3. Obtain the header information for the Earth Engine query and store it in an xarray object.
#    This code does not do a full query for the data (yet)! 

#    In this example, we are just querying some data from Landsat 8 imagery 
#    over a small watershed for demo purposes.
import ee

ee.Initialize()

# Basic cloud masking algorithm
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

Plumas = ee.FeatureCollection("projects/robust-raster/assets/boundaries/Plumas_National_forest")

# This is a dictionary that my code requires. You don't have to touch anything here for demo purposes
# (although it should work with anything, in theory. Feel free to change it if you'd like).
# These parameters get stored and are used to generate the header information needed when we do the full
# run.

"""
earth_engine = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .select(['SR_B4', 'SR_B5'])
    .filterDate('2020-05-01', '2020-08-31')
    .filterBounds(Plumas.geometry())
    .map(prep_sr_l8)
)
"""

vector_path = r"C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\geometry\Plumas_National_forest.geojson"

parameters = {
            'collection': 'LANDSAT/LC08/C02/T1_L2',
            'bands': ['SR_B4', 'SR_B5'],
            'start_date': '2018-05-08',
            'end_date': '2018-05-10',
            'vector_path': vector_path,
            'crs': 'EPSG:3310',
            'scale': 30,
            'map_function': prep_sr_l8
        }

In [ ]:
from robustraster import run

run(
    dataset=None,
    source="ee",
    dataset_params=parameters,
    user_function=compute_ndvi,
    export_params={"flag": "GTiff", "output_folder": "rush123"},
    dask_mode="full"  # for single worker in local test mode
)